# Q3 — train, evaluate, and save automatically

Use the **same existing Colab GPU runtime** in which `Q3 DATALOADER CHECK PASSED` appeared. This notebook assumes the Part 1 environment and the updated `/content/part1` upload already exist. It does not edit your homework files.

Run the preparation cell, complete Drive authorization, then run the long-run cell **once**. It trains from the pretrained BERT checkpoint, backs up the finished model before evaluation, evaluates both test sets, and saves raw predictions before validating them. Leave this cell running while you are away; do not use Run All in your original homework notebook.

The final inspection cell is safe to rerun later. All backups are under `MyDrive/NYU_NLP_HW4/part1/q3/<timestamp>/`. A `COMPLETED` state means training, both evaluations, backups, and basic output checks succeeded; it is not a guarantee of a particular grade.

**Limit:** your training loop only saves model weights at the end. If Colab destroys the runtime during training, the logs and source already saved to Drive survive, but unfinished training weights cannot be recovered by this notebook. It does not add periodic training checkpoints or bypass Colab's runtime limits. If the runtime dies during evaluation, a successfully backed-up model can be used to rerun evaluation without retraining.

## 1. Prepare and mount Drive

Run this first. Finish the Google Drive authorization before starting the long run. The printed backup folder is unique to this run.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import ast
import hashlib
import json
import shutil
import subprocess

# Use the SAME Colab runtime in which your Q3 loader check passed.
PART1 = Path("/content/part1")
PYTHON = "/content/hw4-part1-env/bin/python"
assert Path(PYTHON).is_file(), "Run the Part 1 environment setup first."
for name in ["main.py", "utils.py", "requirements.txt"]:
    assert (PART1 / name).is_file(), f"Upload part1 first: missing {name}"

tree = ast.parse((PART1 / "main.py").read_text())
fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef)
          and n.name == "create_augmented_dataloader")
assert not any(isinstance(n, ast.Name) and n.id == "NotImplementedError"
               for n in ast.walk(fn)), "The uploaded Q3 function is still a placeholder."

gpu = subprocess.run(
    [PYTHON, "-c",
     "import torch; assert torch.cuda.is_available(), 'No GPU'; "
     "print('GPU:', torch.cuda.get_device_name(0))"],
    capture_output=True, text=True,
)
print(gpu.stdout, end="")
if gpu.returncode:
    print(gpu.stderr)
gpu.check_returncode()

# Mount Drive and finish authorization BEFORE starting the long run.
drive.mount("/content/drive")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
Q3_WORK = Path("/content") / f"hw4_q3_{run_id}"
Q3_SAVE = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part1/q3") / run_id
)
Q3_WORK.mkdir(parents=True, exist_ok=False)
(Q3_SAVE / "source").mkdir(parents=True, exist_ok=False)

# Isolated copies prevent stale Q3 files or later edits from affecting this run.
hashes = {}
for name in ["main.py", "utils.py", "requirements.txt"]:
    shutil.copy2(PART1 / name, Q3_WORK / name)
    shutil.copy2(PART1 / name, Q3_SAVE / "source" / name)
    hashes[name] = hashlib.sha256((Q3_WORK / name).read_bytes()).hexdigest()
(Q3_SAVE / "source_sha256.json").write_text(json.dumps(hashes, indent=2))

print("Preparation complete. Training has NOT started.")
print("Working directory:", Q3_WORK)
print("SAVE THIS DRIVE LOCATION:", Q3_SAVE)
print("Run the next cell ONCE to train, evaluate, and back up automatically.")


GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Preparation complete. Training has NOT started.
Working directory: /content/hw4_q3_20261109T201957104463Z
SAVE THIS DRIVE LOCATION: /content/drive/MyDrive/NYU_NLP_HW4/part1/q3/20261109T201957104463Z
Run the next cell ONCE to train, evaluate, and back up automatically.


## 2. Start Q3 once

This is the long-running cell. It handles training, model backup, both evaluations, raw-output backup, and saved accuracy summaries. It prints progress live and writes separate logs to Drive. Do not start a second training command alongside it.

Stages in `status.json`: `training`, `backing_up_checkpoint`, `checkpoint_saved`, `eval_original`, `original_predictions_saved`, `eval_transformed`, and finally `all_results_saved`. If a normal Python exception occurs, it records `FAILED` and attempts to preserve available artifacts. Abrupt runtime termination can leave the last saved state as `RUNNING`; that state alone does not prove the process is still alive.

In [2]:
import json
import subprocess
import traceback
from datetime import datetime, timezone

assert Q3_WORK.is_dir() and Q3_SAVE.is_dir(), "Run the preparation cell first."
assert not (Q3_SAVE / "status.json").exists(), (
    "This run has already started. Inspect its saved status before launching another."
)

record = {
    "run_id": run_id,
    "state": "RUNNING",
    "stage": "starting",
    "checkpoint_backed_up": False,
    "results": {},
    "save_folder": str(Q3_SAVE),
}

def save_status(state, stage):
    record.update(
        state=state,
        stage=stage,
        updated_utc=datetime.now(timezone.utc).isoformat(),
    )
    (Q3_SAVE / "status.json").write_text(json.dumps(record, indent=2))

def back_up_predictions():
    for name in [
        "out_augmented_original.txt",
        "out_augmented_transformed.txt",
    ]:
        if (Q3_WORK / name).is_file():
            shutil.copy2(Q3_WORK / name, Q3_SAVE / name)

def run_stage(stage, *arguments):
    save_status("RUNNING", stage)
    print(f"\n=== {stage.upper()} ===", flush=True)
    print("Drive backup:", Q3_SAVE, flush=True)
    command = [PYTHON, "-u", "main.py", *arguments]
    # The full log goes to Drive while output is also displayed here.
    with (Q3_SAVE / f"{stage}.log").open("x", buffering=1) as log:
        log.write("Command: " + json.dumps(command) + "\n")
        process = subprocess.Popen(
            command, cwd=Q3_WORK,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        try:
            for line in process.stdout:
                log.write(line)
                print(line, end="", flush=True)
            code = process.wait()
            log.write(f"\nExit code: {code}\n")
        except BaseException:
            # Stop only the child process created for this stage.
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
        finally:
            process.stdout.close()
    if code:
        raise RuntimeError(f"{stage} failed with exit code {code}; inspect its saved log.")

def measure_predictions(name):
    values = [int(x) for x in (Q3_WORK / name).read_text().split()]
    assert len(values) == 50000, f"{name}: expected 50,000 integers, got {len(values)}"
    assert set(values) <= {0, 1}, f"{name}: values must be binary"
    predictions, labels = values[0::2], values[1::2]
    accuracy = sum(p == y for p, y in zip(predictions, labels)) / len(labels)
    return {"examples": len(labels), "accuracy": accuracy}, labels

try:
    # This trains a NEW model. It does not retrain Q1 or use the Q1 checkpoint.
    run_stage("training", "--train_augmented")

    # Protect the finished model BEFORE running either evaluation.
    save_status("RUNNING", "backing_up_checkpoint")
    shutil.copytree(Q3_WORK / "out_augmented", Q3_SAVE / "out_augmented")
    record["checkpoint_backed_up"] = True
    save_status("RUNNING", "checkpoint_saved")
    print("\nTRAINED MODEL SAVED TO DRIVE.", flush=True)

    run_stage("eval_original", "--eval", "--model_dir", "out_augmented")
    back_up_predictions()  # Preserve the raw file before judging its format.
    record["results"]["original"], original_labels = measure_predictions(
        "out_augmented_original.txt"
    )
    save_status("RUNNING", "original_predictions_saved")

    run_stage("eval_transformed", "--eval_transformed", "--model_dir", "out_augmented")
    back_up_predictions()
    record["results"]["transformed"], transformed_labels = measure_predictions(
        "out_augmented_transformed.txt"
    )
    assert original_labels == transformed_labels, "Evaluation label ordering differs."

    (Q3_SAVE / "results.json").write_text(json.dumps(record["results"], indent=2))
    save_status("COMPLETED", "all_results_saved")

except BaseException:
    record["error"] = traceback.format_exc()
    # Preserve partial outputs too. They are not automatically valid submissions.
    try:
        back_up_predictions()
        if ((Q3_WORK / "out_augmented").is_dir()
                and not record["checkpoint_backed_up"]):
            shutil.copytree(
                Q3_WORK / "out_augmented", Q3_SAVE / "out_augmented",
                dirs_exist_ok=True,
            )
            record["checkpoint_may_be_incomplete"] = True
    except Exception:
        record["backup_error"] = traceback.format_exc()
    save_status("FAILED", record["stage"])
    print("\nRUN FAILED. Available files and the error are saved at:", Q3_SAVE)
    raise

else:
    print("\nQ3 FINISHED AND BACKED UP:", Q3_SAVE)
    for split, result in record["results"].items():
        print(f"{split}: {result['accuracy']:.2%} ({result['examples']} reviews)")
    print("COMPLETED means the run and format checks passed; inspect the accuracies next.")



=== TRAINING ===
Drive backup: /content/drive/MyDrive/NYU_NLP_HW4/part1/q3/20261109T201957104463Z
/content/hw4-part1-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

Map: 100%|██████████| 25000/25000 [00:23<00:00, 1056.61 examples/s]

Map: 100%|██████████| 25000/25000 [00:24<00:00, 1018.85 examples/s]

Map: 100%|██████████| 50000/50000 [00:48<00:00, 1032.32 examples/s]
Actual training...
len(train_dataloader): 3125
len(eval_dataloader): 3125

Map: 100%|██████████| 5000/5000 [00:02<00:00, 2025.18 examples/s]

Map: 100%|██████████| 30000/30000 [00:29<00:00, 1010.62 examples/s]
Augmented training examples: 30000
Augmented training batches: 3750
Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['

## 3. Inspect after returning

This cell only reads saved files. If you have reconnected to a fresh runtime, you can run this cell on its own after mounting Drive; the Part 1 Python environment is not needed just to read these files. You can also inspect `status.json`, `results.json`, and the logs directly in Google Drive without any Colab runtime.

Do not relaunch training merely because you cannot see the old notebook output. First check whether the model and predictions have already been saved.

In [3]:
# Safe to run after returning. This cell NEVER starts training.
from pathlib import Path
from google.colab import drive
import json

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/NYU_NLP_HW4/part1/q3")
runs = sorted(p for p in root.iterdir() if p.is_dir()) if root.exists() else []
assert runs, "No Q3 backup folder found."
saved_run = runs[-1]  # Change to a specific folder if you have multiple runs.
print("Inspecting:", saved_run)

status_file = saved_run / "status.json"
if status_file.exists():
    status = json.loads(status_file.read_text())
    print(json.dumps(status, indent=2))
else:
    print("Preparation exists, but no saved run status was found.")

print("\nSaved files:")
for path in sorted(saved_run.iterdir()):
    print(path.name)

for name in ["training.log", "eval_original.log", "eval_transformed.log"]:
    path = saved_run / name
    if path.exists():
        print(f"\n--- Last 12 lines of {name} ---")
        print("\n".join(path.read_text(errors="replace").splitlines()[-12:]))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inspecting: /content/drive/MyDrive/NYU_NLP_HW4/part1/q3/20261109T201957104463Z
{
  "run_id": "20261109T201957104463Z",
  "state": "COMPLETED",
  "stage": "all_results_saved",
  "checkpoint_backed_up": true,
  "results": {
    "original": {
      "examples": 25000,
      "accuracy": 0.91844
    },
    "transformed": {
      "examples": 25000,
      "accuracy": 0.89424
    }
  },
  "save_folder": "/content/drive/MyDrive/NYU_NLP_HW4/part1/q3/20261109T201957104463Z",
  "updated_utc": "2026-11-09T23:13:39.998665+00:00"
}

Saved files:
eval_original.log
eval_transformed.log
out_augmented
out_augmented_original.txt
out_augmented_transformed.txt
results.json
source
source_sha256.json
status.json
training.log

--- Last 12 lines of training.log ---
100%|██████████| 11250/11250 [2:21:42<00:00,  1.32it/s, loss=0.00494]
Training completed...
Saving Model....

Exit code: 0